# 11 · Sensibilidad de la regla de interpretación

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Vuelve a interpretar las respuestas guardadas de los 10 modelos con una regla flexible (la emoción en cualquier posición de la respuesta) y la compara con la regla oficial (primera palabra).

*Nombre durante el desarrollo: `07c_Sensibilidad_Parseo` (algunas salidas conservan esa numeración).*

In [1]:
!pip install -q "huggingface_hub<2.0" scikit-learn "pandas<3"

import re, unicodedata
import pandas as pd
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
from sklearn.metrics import accuracy_score, f1_score

login(token=userdata.get('HF_TOKEN'))
REPO_DATOS = "santiagortegasegura/emotion-futbol-data"
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
MODELOS = ["gemma2_9b", "gemma2_2b", "llama_8b", "qwen_3b", "qwen_1_5b", "qwen_7b",
           "llama_3b", "mistral_7b", "phi35_mini", "phi3_mini"]

## 1. Las dos reglas de interpretación

In [2]:
def regla_estricta(generado):
    generado = "" if pd.isna(generado) else str(generado)
    limpio = generado.strip().lower().split()[0] if generado.strip() else ""
    limpio = limpio.strip(".,;:!¡¿?\"'")
    return limpio if limpio in EMOCIONES else "invalido"

def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

def regla_flexible(generado):
    generado = "" if pd.isna(generado) else str(generado)
    for palabra in re.findall(r"\w+", sin_tildes(generado.lower())):
        if palabra in EMOCIONES:
            return palabra
    return "invalido"

for ejemplo in ["burla", "Alegría.", "El comentario expresa enojo", "frustración", "neutral\n\nComentario"]:
    print(f"{ejemplo!r:35} estricta: {regla_estricta(ejemplo):9} flexible: {regla_flexible(ejemplo)}")

'burla'                             estricta: burla     flexible: burla
'Alegría.'                          estricta: invalido  flexible: alegria
'El comentario expresa enojo'       estricta: invalido  flexible: enojo
'frustración'                       estricta: invalido  flexible: invalido
'neutral\n\nComentario'             estricta: neutral   flexible: neutral


## 2. Aplicar las dos reglas a las respuestas guardadas de los 10 modelos

In [3]:
filas, rescatadas = [], []
for modelo in MODELOS:
    ruta = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset",
                           filename=f"final/predicciones/07_predicciones_{modelo}.csv")
    df = pd.read_csv(ruta)
    df['estricta'] = df['prediccion_cruda'].map(regla_estricta)
    df['flexible'] = df['prediccion_cruda'].map(regla_flexible)

    coincide = (df['estricta'] == df['prediccion']).mean()
    fila = {"modelo": modelo, "coincide_con_oficial": round(coincide, 4)}
    for regla in ['estricta', 'flexible']:
        fila[f"acc_{regla}"] = accuracy_score(df['label'], df[regla])
        fila[f"f1_{regla}"] = f1_score(df['label'], df[regla], labels=EMOCIONES, average='macro', zero_division=0)
        fila[f"inval_{regla}_%"] = 100 * (df[regla] == 'invalido').mean()
    cambiadas = df[(df['estricta'] == 'invalido') & (df['flexible'] != 'invalido')]
    fila["rescatadas"] = len(cambiadas)
    fila["rescatadas_correctas_%"] = 100 * (cambiadas['flexible'] == cambiadas['label']).mean() if len(cambiadas) else None
    filas.append(fila)
    rescatadas.append(cambiadas.assign(modelo=modelo))

res = pd.DataFrame(filas)
res["dif_acc_puntos"] = 100 * (res["acc_flexible"] - res["acc_estricta"])
res["dif_f1"] = res["f1_flexible"] - res["f1_estricta"]
pd.set_option("display.width", 200)
print(res[["modelo", "coincide_con_oficial", "acc_estricta", "acc_flexible", "dif_acc_puntos",
           "f1_estricta", "f1_flexible", "inval_estricta_%", "inval_flexible_%",
           "rescatadas", "rescatadas_correctas_%"]].round(4).to_string(index=False))

07_predicciones_gemma2_9b.csv:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

07_predicciones_gemma2_2b.csv:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

07_predicciones_llama_8b.csv:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

07_predicciones_qwen_3b.csv:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

07_predicciones_qwen_1_5b.csv:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

...

07_predicciones_llama_3b.csv:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

07_predicciones_mistral_7b.csv:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

07_predicciones_phi35_mini.csv:   0%|          | 0.00/1.28M [00:00<?, ?B/s]

07_predicciones_phi3_mini.csv:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

    modelo  coincide_con_oficial  acc_estricta  acc_flexible  dif_acc_puntos  f1_estricta  f1_flexible  inval_estricta_%  inval_flexible_%  rescatadas  rescatadas_correctas_%
 gemma2_9b                   1.0        0.8365        0.8400            0.35       0.7881       0.7877              1.69              1.30          39                 89.7436
 gemma2_2b                   1.0        0.8248        0.8280            0.32       0.7762       0.7775              1.69              1.34          35                 91.4286
  llama_8b                   1.0        0.7987        0.8050            0.63       0.7526       0.7528              5.42              4.69          73                 86.3014
   qwen_3b                   1.0        0.7987        0.8052            0.65       0.7506       0.7530              4.93              4.25          68                 95.5882
 qwen_1_5b                   1.0        0.7892        0.7958            0.66       0.7459       0.7484              4.93     

## 3. ¿Cambia el orden de los modelos?

In [4]:
orden = pd.DataFrame({
    "puesto": range(1, len(res) + 1),
    "según regla estricta (oficial)": res.sort_values("f1_estricta", ascending=False)["modelo"].values,
    "según regla flexible": res.sort_values("f1_flexible", ascending=False)["modelo"].values,
})
print(orden.to_string(index=False))

 puesto según regla estricta (oficial) según regla flexible
      1                      gemma2_9b            gemma2_9b
      2                      gemma2_2b            gemma2_2b
      3                       llama_8b              qwen_3b
      4                        qwen_3b             llama_8b
      5                      qwen_1_5b            qwen_1_5b
      6                        qwen_7b              qwen_7b
      7                       llama_3b           mistral_7b
      8                     mistral_7b             llama_3b
      9                     phi35_mini           phi35_mini
     10                      phi3_mini            phi3_mini


## 4. Qué escribieron los modelos cuando la respuesta fue inválida

In [5]:
for modelo in MODELOS:
    df = pd.read_csv(hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset",
                                     filename=f"final/predicciones/07_predicciones_{modelo}.csv"))
    inval = df[df['prediccion'] == 'invalido']['prediccion_cruda'].fillna("").str.strip().str[:40]
    if len(inval) == 0:
        continue
    print(f"\n=== {modelo}: {len(inval)} inválidas ===")
    print(inval.value_counts().head(8).to_string())


=== gemma2_9b: 169 inválidas ===
prediccion_cruda
Emoción: enojo                    7
: enojo\n\nComentario:            4
Emoción: alegria\n\nComentario    3
ción: enojo\n\nComentario         3
.\n\nEmoción: en                  3
: neutral\n\nComentario: El       2
: alegria\n\nComentario: El       2
Emoción: burla                    2

=== gemma2_2b: 169 inválidas ===
prediccion_cruda
Emoción: enojo                 4
: neutral\n\nComentario: El    3
Emoción: burla                 2
: enojo\n\nComentario:         2
: enojo, enojo                 2
.\n\nEmoción: en               2
: alegria, alegria, alegria    2
.\n\nEmoción: alegria          2

=== llama_8b: 542 inválidas ===
prediccion_cruda
oción: enojo        11
Emoción: aleg       10
oción: alegria      10
: alegria\n\nCom     9
Emoción: eno         8
ón: enojo            7
ón: alegria          7
Emoción: a           6

=== qwen_3b: 493 inválidas ===
prediccion_cruda
Emoción: en         11
ón: alegria         10
Emoción: a       

## 5. Guardar

In [6]:
res.round(4).to_csv("07c_sensibilidad_parseo.csv", index=False)
pd.concat(rescatadas)[['modelo', 'text', 'label', 'prediccion_cruda', 'flexible']].to_csv(
    "07c_respuestas_rescatadas.csv", index=False)
print("Descarga 07c_sensibilidad_parseo.csv y 07c_respuestas_rescatadas.csv del panel de la izquierda.")

Descarga 07c_sensibilidad_parseo.csv y 07c_respuestas_rescatadas.csv del panel de la izquierda.
